In [1]:
"""
colour_table.py -- quantile-warped colour ramp breakpoints for p_cal.

WHY NOT LINEAR: p_cal is brutally skewed. Half the bars sit below ~5e-4, 90%
below ~0.26. A linear ramp spends 90% of its colour range on 10% of the bars,
so half a session's markers render identically dark. Log over-corrects the
other way and crushes the top decile, which is the region that must stay
readable.

THE RULE: colour position = fraction of holdout bars at or below this p_cal.
Each 5% of the ramp then carries 5% of the bars. Still a fixed deterministic
function -- same p_cal always gives the same colour -- just warped to the
data's shape instead of the number line.

SHIP: the printed (p_cal, pos) arrays + linear interpolation, byte-identical in
C++ and C#. Same discipline as the isotonic breakpoints in the worker: a small
table plus interp, not a formula each implementation re-tunes by eye.

LOOKUP (both languages, identical):
    pos = interp(p_cal, X[], Y[])        // linear between breakpoints
    pos = clamp(pos, 0, 1)               // below X[0] -> 0, above X[n-1] -> 1
    rgb = ramp(pos)                      // perceptually-uniform, NO green
    if (p_cal < 0) draw nothing          // warm bar, worker sent -1

RAMP: use a Plasma/Inferno-family ramp (dark -> purple -> orange -> yellow).
Turbo puts green mid-scale, which collides with the lamp's GREEN meaning
"safe" at the BOTTOM. Same word, two meanings, unfixable by tuning.
Keep the floor dim rather than black: calm bars recede but stay visible.
"""

import numpy as np
import pandas as pd
from datetime import datetime

In [2]:
import sys
from contextlib import contextmanager

class Tee:
    def __init__(self, *streams): self.streams = streams
    def write(self, text):
        for stream in self.streams: stream.write(text)
        return len(text)
    def flush(self):
        for stream in self.streams: stream.flush()
    def isatty(self): return False

@contextmanager
def copy_output_to(path, mode="a"):
    original_stdout, original_stderr = sys.stdout, sys.stderr

    with open(path, mode, encoding="utf-8", buffering=1) as log:
        sys.stdout = Tee(original_stdout, log)
        sys.stderr = Tee(original_stderr, log)
        try:
            yield
        finally:
            sys.stdout, sys.stderr = original_stdout, original_stderr

In [3]:
# ---------------------------------------------------------------- CONFIG
TEST_FROM = "2026-01-01"
N_POINTS = 50                  # evenly spaced quantile breakpoints
LAMP_GREEN_Q = 0.50            # lamp GREEN upper edge (quantile)
LAMP_RED_Q = 0.95              # lamp RED lower edge (quantile) - see the table in the end of stage 5
DECIMALS = 9                   # printed precision for the table

STAGE0_TAG = 'mnq-6T-3S-9-12am-RSX-vol2-v2'
LOG_FILE = f"logs/colourmap-{STAGE0_TAG}-{int(LAMP_GREEN_Q * 100)}G-{int(LAMP_RED_Q * 100)}R-{datetime.now().strftime('%Y-%m-%d_%H-%M')}.txt"
PRED_PATH = f"model/{STAGE0_TAG}_pred.pqt"
# ----------------------------------------------------------------


def build(p):
    """p -> (X, Y). X = p_cal, Y = exact ECDF (fraction of bars at or below).
    Built on DISTINCT values so tie groups get their true position, not the
    nearest sampled quantile."""
    u, c = np.unique(p, return_counts=True)
    e = np.cumsum(c) / len(p)
    targets = np.unique(np.concatenate(
        [np.linspace(0.0, 1.0, N_POINTS), [LAMP_GREEN_Q, LAMP_RED_Q]]))
    idx = np.searchsorted(e, targets, side="left").clip(0, len(u) - 1)
    idx = np.unique(np.concatenate([[0], idx, [len(u) - 1]]))
    return u[idx], e[idx]


def emit(X, Y):
    xs = ", ".join(f"{v:.{DECIMALS}g}" for v in X)
    ys = ", ".join(f"{v:.6g}" for v in Y)
    n = len(X)

    print(f"\n---- python ----")
    print(f"# MODEL {STAGE0_TAG}")
    print(f"RAMP_X = [{xs}]")
    print(f"RAMP_Y = [{ys}]")

    print(f"\n---- C++ ----")
    print(f"// MODEL {STAGE0_TAG}")
    print(f"static const int    RAMP_N = {n};")
    print(f"static const double RAMP_X[RAMP_N] = {{ {xs} }};")
    print(f"static const double RAMP_Y[RAMP_N] = {{ {ys} }};")

    print(f"\n---- C# ----")
    print(f"// MODEL {STAGE0_TAG}")
    print(f"static readonly double[] RampX = {{ {xs} }};")
    print(f"static readonly double[] RampY = {{ {ys} }};")


def verify(p, X, Y):
    pos = np.interp(p, X, Y)
    print(f"\n---- flatness check ({len(p)} holdout bars) ----")
    hist, _ = np.histogram(pos, bins=10, range=(0.0, 1.0))
    for i, c in enumerate(hist):
        print(f"  ramp {i/10:.1f}-{(i+1)/10:.1f}  {c/len(p):6.2%} of bars"
              f"   {'#' * int(round(60 * c / len(p) * 10))}")
    print("  (each band should hold ~10%; the bottom band absorbs the "
          "tie-heavy low tail)")

    print(f"\n---- cross-implementation check values ----")
    print(f"  {'p_cal':>14s}  {'ramp pos':>9s}   note")
    for q, note in [(0.0, "min"), (LAMP_GREEN_Q, "lamp GREEN edge"),
                    (0.75, ""), (0.90, ""), (0.92, ""),
                    (LAMP_RED_Q, "lamp RED edge"), (0.98, ""), (1.0, "max")]:
        v = float(np.quantile(p, q))
        print(f"  {v:14.9g}  {float(np.interp(v, X, Y)):9.6f}   {note}")
    print("  all three implementations must agree on these RGB values")

with copy_output_to(LOG_FILE):
    
    # ---------------------------------------------------------------- run
    pred = pd.read_parquet(PRED_PATH)
    h = pred[pred["timestamp"] >= TEST_FROM]
    p = h["p_cal"].to_numpy(np.float64)
    print(f"{PRED_PATH}\n{len(p)} holdout bars from {TEST_FROM}")
    
    X, Y = build(p)
    print(f"\n{len(X)} breakpoints (from {N_POINTS} quantiles + lamp cuts, "
          f"ties collapsed)")
    print(pd.DataFrame({"p_cal": X, "ramp_pos": Y}).to_string(index=False))
    
    emit(X, Y)
    verify(p, X, Y)

model/mnq-6T-3S-9-12am-RSX-vol2-v2_pred.pqt
516883 holdout bars from 2026-01-01

34 breakpoints (from 50 quantiles + lamp cuts, ties collapsed)
   p_cal  ramp_pos
0.000000  0.031378
0.000071  0.062161
0.000084  0.139682
0.000097  0.205979
0.000272  0.310676
0.000543  0.331450
0.000597  0.406624
0.000656  0.420292
0.001336  0.440154
0.001738  0.488774
0.002126  0.545731
0.002893  0.564642
0.005910  0.586144
0.007355  0.600115
0.008127  0.624033
0.009819  0.672680
0.012411  0.673969
0.020245  0.700572
0.027555  0.718495
0.038295  0.737774
0.046384  0.760896
0.077107  0.784568
0.090589  0.797237
0.131579  0.817831
0.179896  0.840834
0.250000  0.857300
0.326009  0.879423
0.452323  0.898000
0.601610  0.920301
0.748107  0.941358
0.825656  0.950472
0.878788  0.959223
0.974900  0.981305
1.000000  1.000000

---- python ----
# MODEL mnq-6T-3S-9-12am-RSX-vol2-v2
RAMP_X = [0, 7.09320448e-05, 8.36143736e-05, 9.72857306e-05, 0.000272405334, 0.000543065078, 0.000596676487, 0.000656167977, 0.001336302